# Run & verify

**Is the deployment sound, can I run a scan, and are the tables consistent?**

The only notebook here that writes anything. The other six read what this one produced.

## How to read this notebook

Every cell answers one question and shows one thing. Run them in order the first time; after
that any cell can be re-run on its own.

**Set the widgets at the top before you run anything.** `catalog` has no default on purpose.
Set the notebook to **Run accessed commands** (the dropdown beside *Run all*) if you want a
widget change to re-run the cells that depend on it — otherwise you will change the filter and
read a chart drawn under the old one.

Everything here reads one scan, pinned in cell 1. Charts that span scans say so in their title.

In [ ]:
PAGE = {"disappearance": ("scan_ts", ["scan_ts", "midpoint"])}

import os, sys

_paths = []
try:
    _paths.append(dbutils.widgets.get("module_path"))
except Exception:  # noqa: BLE001 -- the widget does not exist yet on a first run
    pass
_here = os.getcwd()
_paths += [_here, os.path.dirname(_here)]
for _p in _paths:
    if _p and os.path.exists(os.path.join(_p, "panels.py")):
        sys.path.insert(0, _p)
        break
else:
    raise RuntimeError("brick modules are not on sys.path -- see brick/README.md, step 2")

import panels, figures, tiles

panels.declare_widgets(**PAGE)
ctx = panels.context(spark, **{name: str(spec[0]) for name, spec in PAGE.items()})
displayHTML(tiles.scan_zone_from(panels.last_scan(spark, ctx).first()))

## Is the deployment sound?

Every module must report the same version **from the same folder**. A half-updated folder
imports cleanly and then fails much later at something that looks unrelated — the first v2
run hit exactly that, 137,870 findings in, as a schema mismatch that named neither the
stale file nor the fix.

The printed path is the other half: `config`, `metrics` and `ingest` are generic module
names, and so are `panels`, `figures` and `tiles`. If something else on `sys.path` shadows
one, the path below says so — an `AttributeError` three cells later would not.

In [ ]:
import config, dbx, ingest, ledger, metrics, run_pipeline

for _m in (config, dbx, ingest, ledger, metrics, run_pipeline, panels, figures, tiles):
    print(f"{_m.__name__:14} {getattr(_m, 'MODULE_VERSION', 'PRE-2.0 — STALE'):8}"
          f" {_m.__file__}")

## Run a scan

Needs credentials in a secret scope and write access to the schema. Set `catalog`,
`schema`, `wiz_api_url` and `secret_scope` in the widgets first.

Safe to re-run: the scan id is the idempotency guard, so a retry with the same id is a
no-op rather than a double count. `--rebuild_ledger` replays every archived scan and is
the one thing here that is not cheap.

**After editing or re-pasting any module, restart Python — do not reload.**
`dbutils.library.restartPython()`. `importlib.reload` is worse than doing nothing: it
re-executes a module while every other module still holds references into the old one.

In [ ]:
from run_pipeline import main

# **The widgets decide, and cell 1 already resolved them.** `sys.argv` is cleared rather than
# populated: `run_pipeline.param` reads argv *before* the widgets, so passing anything here
# would let this cell write somewhere cell 1 did not look -- which is exactly what happened
# before. `panels.context` above called `ensure_tables()` against `ctx.tables`, so a
# disagreement means two registers, one of them empty and in the wrong place.
#
# Set `data_path` for a directory-backed register, or `catalog` / `schema` for a
# catalog-backed one. Set `csv_path` and the scan exports itself afterwards -- which is what
# makes its results survive when the Delta side is somewhere ephemeral.
import sys
sys.argv = [sys.argv[0]]

print("register :", ctx.tables.ledger)
print("csv       :", ctx.param("csv_path") or "(not exporting -- set the csv_path widget)")

result = main()
print(result)

## Are the tables consistent?

The three gold tables and the run log should agree on which scan is the latest. They
disagree when a run died between two writes; the pipeline refuses to start in that state,
and this surfaces it *before* the next run hits it — and before somebody reads a page
whose halves came from different scans.

The ledger is the exception by design: it is MERGEd current state and carries
`last_scan_id` rather than `scan_id`.

In [ ]:
display(panels.scan_pin_check(spark, ctx))

In [ ]:
display(panels.table_inventory(spark, ctx))

In [ ]:
display(panels.run_health(spark, ctx))

---

GAS's *Data* page also imports a legacy migration bundle. brick ingests from the Wiz API
and has no import path, so there is nothing to expose. To get data out, use the download
button on any result grid above.